In [2]:
import zipfile

import os

# Sostituisci con il percorso del tuo file .zip

file_compresso = 'key_results_testi.zip'

# Sostituisci con la cartella di destinazione

cartella_destinazione = 'NLP'

# Estrae l'intero contenuto

with zipfile.ZipFile(file_compresso, 'r') as zip_ref:

    zip_ref.extractall(cartella_destinazione)

    print("Estrazione completata con successo!")



Estrazione completata con successo!


In [4]:
# 2.1 Funzione per estrarre metadati dal nome del file
import re
import pandas as pd
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")

    # Cerca il pattern del periodo: Mese_Anno_-_Mese_Anno (es. Apr_2016_-_Dec_2016)
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')  # es. "Apr 2016"
        fine_periodo = periodo_match.group(2).replace('_', ' ')    # es. "Dec 2016"
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base.replace('_', ' ')

    # Paese: tutto quello che precede il primo mese o anno
    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base.replace('_', ' ')

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,
        "fine_periodo": fine_periodo,
        "nome_file": nome_file
    }

# 2.2 Caricamento e pulizia dei report
files = [f for f in os.listdir("key_results_testi") if f.endswith(".txt")]
print(f"Trovati {len(files)} file .txt da elaborare.")

testi_puliti = []
metadata_list = []

for nome_file in files:
    path = os.path.join("key_results_testi", nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo_completo = f.read()

    # Rimuovi l'intestazione dei metadati (tutto ciò che precede la riga divisoria di '=' o '-')
    testo_pulito = re.sub(r'^.*?={20,}\n*', '', testo_completo, flags=re.DOTALL).strip()

    if not testo_pulito:
        print(f"Attenzione: il file {nome_file} è vuoto dopo la pulizia.")
        continue

    testi_puliti.append(testo_pulito)
    metadata_list.append(estrai_metadata(nome_file))

# Creazione del DataFrame
df = pd.DataFrame(metadata_list)
df["testo"] = testi_puliti
print(f"Caricati con successo {len(df)} report nel DataFrame.")
df.head()

Trovati 497 file .txt da elaborare.
Caricati con successo 497 report nel DataFrame.


,paese,periodo,inizio_periodo,fine_periodo,nome_file,testo
0,Gaza Strip,Sep 2024 / Apr 2025,Sep 2024,Apr 2025,Gaza_Strip_Sep_2024_-_Apr_2025_KeyResults.txt,"One year into the conflict, the risk of Famine..."
1,Afghanistan,Apr 2020 / Nov 2020,Apr 2020,Nov 2020,Afghanistan_Apr_2020_-_Nov_2020_KeyResults.txt,Food insecurity remains alarmingly high in Afg...
2,Afghanistan,Nov 2017 / Feb 2018,Nov 2017,Feb 2018,Afghanistan_Nov_2017_-_Feb_2018_KeyResults.txt,"During the 2017 post-harvest season, 33% of th..."
3,South Sudan,Sep 2018 / Mar 2019,Sep 2018,Mar 2019,South_Sudan_Sep_2018_-_Mar_2019_KeyResults.txt,"Based on the September IPC analysis, it is exp..."
4,Mozambique,Jun 2020 / Sep 2020,Jun 2020,Sep 2020,Mozambique_Jun_2020_-_Sep_2020_KeyResults.txt,The results of this Acute Food Insecurity pilo...


In [5]:
import numpy as np
from llama_cpp import Llama
from sklearn.metrics.pairwise import cosine_similarity

# 1. Configurazione del Modello per Apple Silicon (M4 Max)
MODEL_PATH = "./gte-qwen2-7b-instruct-q8_0.gguf" # Inserisci il percorso reale del modello scaricato

print("Inizializzazione modello su backend Metal...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1, # -1 forza lo scaricamento di tutti i layer (pesi) sulla GPU
    n_ctx=8192,      # Alloca memoria sufficiente per i tuoi documenti da 1800 token
    embedding=True,  # Cruciale: disabilita la generazione testo, abilita il calcolo vettoriale
    verbose=False    # Spegne i log di debug di basso livello
)

# 2. Struttura del Prompt per Clustering Simmetrico
INSTRUCTION = "Instruct: Given a text, retrieve texts that share similar drivers and causes of food insecurity.\nQuery: "

def get_embedding(text: str) -> np.ndarray:
    """
    Formatta il documento e calcola l'embedding vettoriale.
    """
    formatted_input = INSTRUCTION + text

    # Esegue l'inferenza. L'oggetto restituito è un dizionario
    response = llm.create_embedding(formatted_input)

    # Estrae la lista di float e la converte in array NumPy
    embedding = np.array(response["data"][0]["embedding"])
    return embedding

# 3. Pipeline di Elaborazione Documenti
# Sostituisci questa lista con il caricamento del tuo dataset reale

print(f"Calcolo embedding per {len(testi_puliti)} documenti...")

# Generazione della matrice degli embedding (Documenti x Dimensioni Vettoriali)
embeddings_matrix = np.array([get_embedding(doc) for doc in testi_puliti])

print(f"Shape della matrice degli embedding: {embeddings_matrix.shape}")

# 4. Verifica di Similarità (Test per il Clustering)
# Se il prompt istruzionale funziona, la distanza tra i doc 1 e 4 (Economia)
# deve essere minore (similarità maggiore) rispetto ai doc 1 e 3 (AI/NLP)

similarity_matrix = cosine_similarity(embeddings_matrix)

print("\n--- Test di Similarità Semantica ---")
print(f"Doc 1 (Finance) vs Doc 4 (Economy): {similarity_matrix[0][3]:.4f}")
print(f"Doc 1 (Finance) vs Doc 3 (NLP):     {similarity_matrix[0][2]:.4f}")

Inizializzazione modello su backend Metal...


ValueError: Model path does not exist: ./gte-qwen2-7b-instruct-q8_0.gguf

In [11]:
import os
print("Working Directory attuale:", os.getcwd())
print("File effettivamente visibili qui:", os.listdir('.'))

Working Directory attuale: /Users/niki/PycharmProjects/HERO/hero_v6/NLP
File effettivamente visibili qui: ['Embedding1.ipynb', 'key_results_testi.zip', '.DS_Store', 'Clustering_Food_Insecurity.ipynb', 'bertopic_food_insecurity.ipynb', 'key_results_testi', 'Embedding_denso.ipynb', 'visualizzazioni_verify', 'LLM_prova.ipynb', 'nlp_llama.ipynb', 'NLP.ipynb', 'prova', 'Copia_di_Untitled0.ipynb']


In [13]:
import os
import numpy as np
from llama_cpp import Llama
from sklearn.cluster import KMeans

# ==========================================
# 1. CONFIGURAZIONE HARDWARE E MODELLO
# ==========================================
# Sostituisci con il percorso reale del tuo file GGUF
MODEL_PATH = "../../gte-Qwen2-7B-instruct.Q8_0.gguf"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Modello non trovato in: {MODEL_PATH}. "
        f"Scarica il file GGUF prima di avviare lo script."
    )

print("Inizializzazione del modello con accelerazione Metal su M4 Max...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1, # Carica tutti i layer sulla GPU Metal
    n_ctx=8192,      # Supporta i tuoi documenti da 1800 token
    embedding=True,  # Abilita la generazione di vettori
    verbose=False    # Disattiva i log ridondanti di llama.cpp
)

# ==========================================
# 2. INGEGNERIZZAZIONE DEL PROMPT (FOCUS DRIVERS)
# ==========================================
# Questo prompt forza l'attention mechanism a pesare le cause della food insecurity
INSTRUCTION = "Instruct: Given a text, retrieve texts that share similar drivers and causes of food insecurity.\nQuery: "

def generate_document_embedding(text: str) -> np.ndarray:
    """
    Applica il prompt di sistema al documento e genera l'embedding.
    """
    formatted_input = INSTRUCTION + text
    response = llm.create_embedding(formatted_input)
    return np.array(response["data"][0]["embedding"])

# ==========================================
# 3. DATASET DI TEST (DOCUMENTI SULLA FOOD INSECURITY)
# ==========================================
# Esempi con cause diverse: siccità/clima, conflitti armati, barriere economiche

# ==========================================
# 4. GENERAZIONE DELLA MATRICE DI EMBEDDING
# ==========================================
print(f"\nGenerazione degli embedding per {len(testi_puliti)} documenti...")
embeddings = []
for i, doc in enumerate(testi_puliti):
    print(f"Elaborazione documento {i+1}/{len(testi_puliti)}...")
    embeddings.append(generate_document_embedding(doc))

embeddings_matrix = np.array(embeddings)
print(f"Matrice degli embedding creata. Dimensioni: {embeddings_matrix.shape}")

Inizializzazione del modello con accelerazione Metal su M4 Max...

Generazione degli embedding per 497 documenti...
Elaborazione documento 1/497...
Elaborazione documento 2/497...
Elaborazione documento 3/497...
Elaborazione documento 4/497...
Elaborazione documento 5/497...
Elaborazione documento 6/497...
Elaborazione documento 7/497...
Elaborazione documento 8/497...
Elaborazione documento 9/497...
Elaborazione documento 10/497...
Elaborazione documento 11/497...
Elaborazione documento 12/497...
Elaborazione documento 13/497...
Elaborazione documento 14/497...
Elaborazione documento 15/497...
Elaborazione documento 16/497...
Elaborazione documento 17/497...
Elaborazione documento 18/497...
Elaborazione documento 19/497...
Elaborazione documento 20/497...
Elaborazione documento 21/497...
Elaborazione documento 22/497...
Elaborazione documento 23/497...
Elaborazione documento 24/497...
Elaborazione documento 25/497...
Elaborazione documento 26/497...
Elaborazione documento 27/497...
El

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (497,) + inhomogeneous part.